In [1]:
import json
from datetime import datetime

def calculate(expression:str):
    allowedChars="0123456789+-%/*(). "
    for char in expression:
        if char not in allowedChars:
            return "Invalid Input. Alphabets not allowed. Enter Numbers."
    storeItems=[]
    i=0
    while i<len(expression):
        if expression[i]==" ":
            i+=1
        elif expression[i] in "0123456789.":
            specificNo=""
            while i < len(expression) and expression[i] in "0123456789.":
                specificNo+=expression[i]
                i+=1
            try:
                if "." in specificNo:
                    number=float(specificNo)
                else:
                    number=int(specificNo)
                storeItems.append(number)
            except ValueError:
                return "Invalid Expression Format"
        elif expression[i:i+2]=="**":
            storeItems.append("**")
            i+=2
        elif expression[i]=="-" and (i==0 or expression[i-1] in "+-*/%("):
            #for negative values first
            i+=1
            specificNo="-"
            while i<len(expression) and expression[i] in "0123456789.":
                specificNo+=expression[i]
                i+=1
            try:
                if "." in specificNo:
                    number=float(specificNo)
                else:
                    number=int(specificNo)
                storeItems.append(number)
            except ValueError:
                return "Invalid Expression Format"
        else:
            storeItems.append(expression[i])
            i+=1

    #BODMAS for ()
    while "(" in storeItems:
        begin=0
        for j in range (len(storeItems)):
            if storeItems[j]=="(":
                begin=j
        end=begin
        while end<len(storeItems) and storeItems[end]!=")":
            end+=1
        if end>=len(storeItems):
            return "Invalid Expression"
        subExpression=storeItems[begin+1:end]
        if not subExpression:
            return "Empty Brackets without values cannot be accepted"
        for groupOfOper in [["**"], ["/", "*", "%"], ["+", "-"]]:
            j=0
            while j<len(subExpression):
                if subExpression[j] in groupOfOper:
                    if j-1<0 or j+1>=len(subExpression):
                        return "Invalid Expression"
                    left=subExpression[j-1]
                    operator=subExpression[j]
                    right=subExpression[j+1]
                    if operator=="/" and right==0:
                        return "Division by Zero not possible"
                    if operator=="%" and right==0:
                        return "Modulo by Zero not possible"
                    if operator=="**":
                        result=left**right
                    elif operator=="/":
                        result=left/right
                    elif operator=="*":
                        result=left*right
                    elif operator=="%":
                        result=left%right
                    elif operator=="+":
                            result=left+right
                    elif operator=="-":
                            result=left-right
                    subExpression[j-1:j+2]=[result]
                else:
                    j+=1
        storeItems[begin:end+1]=subExpression
            
    #BODMAS for without ()    
    for groupOfOper in [["**"], ["/", "*", "%"], ["+", "-"]]:
        j=0
        while j<len(storeItems):
            if storeItems[j] in groupOfOper:
                if j-1<0 or j+1>=len(storeItems):
                    return "Invalid Expression"
                left=storeItems[j-1]
                operator=storeItems[j]
                right=storeItems[j+1]
                if operator=="/" and right==0:
                    return "Division by Zero not possible"
                if operator=="%" and right==0:
                    return "Modulo by Zero not possible"
                if operator=="**":
                    result=left**right
                elif operator=="/":
                    result=left/right
                elif operator=="*":
                    result=left*right
                elif operator=="%":
                    result=left%right
                elif operator=="+":
                        result=left+right
                elif operator=="-":
                        result=left-right
                storeItems[j-1:j+2]=[result]
            else:
                j+=1
    if len(storeItems)!=1:
        return "Invalid Expression"
    return storeItems[0]

In [2]:
def calculator_tool(expression):
    return calculate(expression)

In [3]:
print(calculator_tool("25*8"))
print(calculator_tool("100/4"))
print(calculator_tool("(10+5)*2"))

200
25.0
30


In [4]:
import chromadb
from sentence_transformers import SentenceTransformer
import os

In [5]:
embedding_model=SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
documents_folder="documents"
documents={}

for filename in os.listdir(documents_folder):
    if filename.endswith(".txt"):
        file_path=os.path.join(documents_folder,filename)
        with open(file_path,"r",encoding="utf-8") as file:
            documents[filename]=file.read()

In [7]:
chunks=[]

for filename,content in documents.items():
    sentences=content.replace("\n"," ").split(". ")

    for i in range(0,len(sentences),3):
        chunk=". ".join(sentences[i:i+3]).strip()

        if chunk and not chunk.endswith("."):
            chunk+="."

        chunks.append({
            "file":filename,
            "text":chunk
        })

In [8]:
documents_list=[chunk["text"] for chunk in chunks]
embeddings=embedding_model.encode(documents_list)

In [9]:
client=chromadb.Client()

collection=client.create_collection(name="agent_documents")

ids=[str(i) for i in range(len(chunks))]
metadatas=[{"source":chunk["file"]} for chunk in chunks]

collection.add(
    ids=ids,
    documents=documents_list,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

In [10]:
def document_search_tool(query):
    query_embedding=embedding_model.encode([query])

    results=collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=3
    )

    search_results=[]

    for i in range(len(results["documents"][0])):
        search_results.append({
            "source":results["metadatas"][0][i]["source"],
            "content":results["documents"][0][i]
        })

    return search_results

In [11]:
results=document_search_tool("What is deep learning?")

for result in results:
    print(result["source"])
    print(result["content"])
    print("-"*50)

deep_learning.txt
Deep Learning is a specialized branch of Machine Learning that uses artificial neural networks to learn complex patterns from large amounts of data.  Deep learning models are inspired by the structure of the human brain and contain multiple layers of interconnected neurons.  Neural networks consist of input layers, hidden layers, and output layers that process information and generate predictions.
--------------------------------------------------
deep_learning.txt
Deep Learning is commonly used in image recognition, natural language processing, speech recognition, and autonomous systems.  Popular deep learning frameworks include TensorFlow, PyTorch, and Keras.  Deep learning requires large datasets and significant computational power, but it can achieve highly accurate results in complex tasks.
--------------------------------------------------
ai.txt
Machine Learning is a major subset of Artificial Intelligence where computers learn patterns from data instead of bei

In [12]:
def file_reader_tool(filename):
    file_path=os.path.join(documents_folder,filename)

    if not os.path.exists(file_path):
        return "File not found."

    with open(file_path,"r",encoding="utf-8") as file:
        return file.read()

In [13]:
print(file_reader_tool("ai.txt"))

Artificial Intelligence (AI) is a branch of computer science that focuses on creating systems capable of performing tasks that normally require human intelligence.

AI systems can learn from data, recognize patterns, understand natural language, make decisions, and solve complex problems.

Artificial Intelligence is used in many fields including healthcare, finance, education, cybersecurity, robotics, and autonomous vehicles.

Machine Learning is a major subset of Artificial Intelligence where computers learn patterns from data instead of being explicitly programmed for every task.

Generative AI is a type of Artificial Intelligence that can create new content such as text, images, audio, and code using advanced models.


In [14]:
from groq import Groq

In [15]:
client=Groq(api_key="your_API_key")

In [16]:
def decide_tool(user_task):
    prompt=f"""
You are an AI agent that selects the correct tool for a user task.

Available tools:
1. Calculator: Use for mathematical calculations.
2. Document Search: Use to answer questions using information from documents.
3. File Reader: Use when the user asks to read or retrieve the contents of a specific file.

User Task:
{user_task}

Respond in exactly this format:
Tool: Calculator
Input: 25*8

OR

Tool: Document Search
Input: What is deep learning?

OR

Tool: File Reader
Input: ai.txt
"""

    response=client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ]
    )

    return response.choices[0].message.content

In [17]:
print(decide_tool("Calculate 25*8"))

Tool: Calculator
Input: 25*8


In [18]:
print(decide_tool("What is deep learning?"))

Tool: Document Search
Input: What is deep learning?


In [19]:
print(decide_tool("Read the contents of ai.txt"))

Tool: File Reader
Input: ai.txt


In [20]:
def run_tool(user_task):
    decision=decide_tool(user_task)

    print("Agent Decision:")
    print(decision)

    if "Tool: Calculator" in decision:
        tool_input=decision.split("Input:")[1].strip()
        return calculator_tool(tool_input)

    elif "Tool: Document Search" in decision:
        tool_input=decision.split("Input:")[1].strip()
        return document_search_tool(tool_input)

    elif "Tool: File Reader" in decision:
        tool_input=decision.split("Input:")[1].strip()
        return file_reader_tool(tool_input)

    return "No suitable tool found."

In [21]:
result=run_tool("Calculate 25*8")
print(result)

Agent Decision:
Tool: Calculator
Input: 25*8
200


In [22]:
result=run_tool("What is deep learning?")
print(result)

Agent Decision:
Tool: Document Search
Input: What is deep learning?
[{'source': 'deep_learning.txt', 'content': 'Deep Learning is a specialized branch of Machine Learning that uses artificial neural networks to learn complex patterns from large amounts of data.  Deep learning models are inspired by the structure of the human brain and contain multiple layers of interconnected neurons.  Neural networks consist of input layers, hidden layers, and output layers that process information and generate predictions.'}, {'source': 'deep_learning.txt', 'content': 'Deep Learning is commonly used in image recognition, natural language processing, speech recognition, and autonomous systems.  Popular deep learning frameworks include TensorFlow, PyTorch, and Keras.  Deep learning requires large datasets and significant computational power, but it can achieve highly accurate results in complex tasks.'}, {'source': 'ai.txt', 'content': 'Machine Learning is a major subset of Artificial Intelligence wher

In [23]:
result=run_tool("Read the contents of ai.txt")
print(result)

Agent Decision:
Tool: File Reader
Input: ai.txt
Artificial Intelligence (AI) is a branch of computer science that focuses on creating systems capable of performing tasks that normally require human intelligence.

AI systems can learn from data, recognize patterns, understand natural language, make decisions, and solve complex problems.

Artificial Intelligence is used in many fields including healthcare, finance, education, cybersecurity, robotics, and autonomous vehicles.

Machine Learning is a major subset of Artificial Intelligence where computers learn patterns from data instead of being explicitly programmed for every task.

Generative AI is a type of Artificial Intelligence that can create new content such as text, images, audio, and code using advanced models.


In [24]:
def generate_final_response(user_task,tool_result):
    prompt=f"""
You are an AI assistant.

User Task:
{user_task}

Tool Result:
{tool_result}

Use the tool result to answer the user's task clearly and accurately.
Do not mention internal tool selection.
"""

    response=client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ]
    )

    return response.choices[0].message.content

In [25]:
user_task="Calculate 25*8"

tool_result=run_tool(user_task)

final_response=generate_final_response(user_task,tool_result)

print(final_response)

Agent Decision:
Tool: Calculator
Input: 25*8
To calculate 25 times 8, we get the result: 200. This is the product of 25 multiplied by 8.


In [26]:
user_task="What is deep learning?"

tool_result=run_tool(user_task)

final_response=generate_final_response(user_task,tool_result)

print(final_response)

Agent Decision:
Tool: Document Search
Input: What is deep learning?
Deep learning is a specialized branch of machine learning that uses artificial neural networks to learn complex patterns from large amounts of data. It is inspired by the structure of the human brain and contains multiple layers of interconnected neurons. 

A deep learning model typically consists of input layers, hidden layers, and output layers that process information and generate predictions.

Deep learning is commonly used in various applications, including:

1. Image recognition: where deep learning models can recognize and classify images with high accuracy.
2. Natural language processing: where deep learning models can process and understand human language, enabling tasks such as language translation, text summarization, and sentiment analysis.
3. Speech recognition: where deep learning models can recognize and understand spoken language, enabling applications such as voice assistants and speech-to-text systems

In [27]:
user_task="Read the contents of ai.txt"

tool_result=run_tool(user_task)

final_response=generate_final_response(user_task,tool_result)

print(final_response)

Agent Decision:
Tool: File Reader
Input: ai.txt
Based on the contents of the file ai.txt, here are the details:

Artificial Intelligence (AI) is a branch of computer science that focuses on creating systems capable of performing tasks that normally require human intelligence. These tasks include learning from data, recognizing patterns, understanding natural language, making decisions, and solving complex problems.

AI is a versatile technology used in various fields, such as:

- Healthcare
- Finance
- Education
- Cybersecurity
- Robotics
- Autonomous vehicles

Additionally, Machine Learning is a significant subset of Artificial Intelligence. It enables computers to learn from data patterns instead of being explicitly programmed for every task.

There's also a type of Artificial Intelligence called Generative AI, which allows models to create new content, including text, images, audio, and code.


In [28]:
while True:
    user_task=input("Enter your task or type exit: ")

    if user_task.lower()=="exit":
        break

    tool_result=run_tool(user_task)
    final_response=generate_final_response(user_task,tool_result)

    print("\nFinal Response:")
    print(final_response)
    print("-"*50)

Enter your task or type exit:  calculatoe


Agent Decision:
Tool: Calculator
Input: 25*8

Final Response:
The task is to calculate something. Based on the result provided (200), I will assume a simple multiplication task. If you want to calculate something else, please clarify.

The task "calculatoe" seems to be incomplete or inaccurate, possibly containing a typo. However, considering a possible calculation task "calculate", the tool result 200 could be interpreted as follows:

If you wanted to calculate 2 * 100, the result would indeed be 200.
--------------------------------------------------


Enter your task or type exit:  calculate 45*0


Agent Decision:
Tool: Calculator
Input: 45*0

Final Response:
To calculate 45 multiplied by 0, the result is directly 0.

This is because any number multiplied by 0 equals 0. In this case, 45 multiplied by 0 also equals 0.

So, the final result of the calculation is 0.
--------------------------------------------------


Enter your task or type exit:  what is ai


Agent Decision:
Tool: Document Search
Input: What is artificial intelligence?

Final Response:
Artificial Intelligence (AI) is a branch of computer science that focuses on creating systems capable of performing tasks that normally require human intelligence. AI systems can learn from data, recognize patterns, understand natural language, make decisions, and solve complex problems. They are used in various fields, such as:

1. Healthcare
2. Finance
3. Education
4. Cybersecurity
5. Robotics
6. Autonomous vehicles

AI has several subsets, including Machine Learning, which is a major subset where computers learn patterns from data instead of being explicitly programmed for every task. Generative AI is a type of AI that can create new content, such as text, images, audio, and code, using advanced models.

Machine Learning is a field of AI that enables computers to learn from data and improve their performance without being explicitly programmed. It uses algorithms to identify patterns in da

Enter your task or type exit:  exit
